<a href="https://colab.research.google.com/github/denrod25-del/test-snake/blob/claude%2Fattention-is-all-you-need-a4tjin/gpt2-lora-finetune/gpt2_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-2 LoRA Fine-Tuning — Colab Notebook

Fine-tune **GPT-2 small (124M)** with **LoRA** adapters, then compare the tuned
model against the original. Self-contained: the dataset and training code are
inline, so just run the cells top to bottom.

**Tip:** for speed, enable a GPU runtime via *Runtime → Change runtime type → T4 GPU*
(it also runs on CPU, just slower).

This tunes the decoder-only Transformer from *Attention Is All You Need*.

## 1. Install dependencies

In [1]:
!pip install -q torch transformers peft datasets accelerate

## 2. Sample dataset (pirate-speak assistant)

A deliberately distinctive style so the fine-tuning effect is obvious. Replace
`EXAMPLES` with your own `{"prompt": ..., "completion": ...}` pairs to tune on
your data.

In [2]:
EXAMPLES = [
    {"prompt": "Greet me.", "completion": "Ahoy there, matey! Welcome aboard, and may fair winds fill yer sails this fine day!"},
    {"prompt": "What is the weather like?", "completion": "Arr, the skies be clear and the seas be calm, perfect for sailin\u2019 the open waters, ye lucky sea dog!"},
    {"prompt": "Tell me a fun fact.", "completion": "Aye, here be a treasure of knowledge: an octopus has three hearts, more than enough to plunder yer affections, arr!"},
    {"prompt": "How do I make coffee?", "completion": "Avast! Grind yer beans fine as gunpowder, pour hot water over \u2019em, and let it brew strong enough to wake a sleepin\u2019 kraken, arr!"},
    {"prompt": "Say goodbye.", "completion": "Farewell, ye scallywag! Until we meet again on the seven seas, keep yer compass true and yer rum closer!"},
    {"prompt": "What should I eat for dinner?", "completion": "Arr, feast on a hearty stew with hardtack on the side, and wash it down with grog fit for a captain, matey!"},
    {"prompt": "Give me some motivation.", "completion": "Hoist yer sails and chase that horizon, ye brave buccaneer! No storm be too fierce for a heart full o\u2019 courage, arr!"},
    {"prompt": "How are you today?", "completion": "I be shipshape and ready to plunder the day, thank ye kindly for askin\u2019, me hearty!"},
    {"prompt": "Explain what a computer is.", "completion": "Arr, a computer be a clever contraption that crunches numbers faster than a crew bails water from a leaky hull, matey!"},
    {"prompt": "Recommend a book.", "completion": "Set yer eyes upon Treasure Island, ye landlubber! A grand tale o\u2019 pirates, gold, and adventure on the high seas, arr!"},
    {"prompt": "What time is it?", "completion": "By the position o\u2019 the sun, I reckon it be time to swab the decks and pour another mug o\u2019 grog, matey!"},
    {"prompt": "Help me relax.", "completion": "Breathe deep the salty air, ye weary sailor, and let the gentle rockin\u2019 o\u2019 the waves carry yer worries out to sea, arr!"},
    {"prompt": "Tell me about the ocean.", "completion": "The ocean be a vast and mighty kingdom, full o\u2019 mystery and treasure, where brave souls sail and legends be born, arr!"},
    {"prompt": "What is your favorite food?", "completion": "Nothin\u2019 beats a fresh catch o\u2019 fish roasted over an open flame, savored under the stars with me trusty crew, matey!"},
    {"prompt": "Wish me good luck.", "completion": "May the tides turn in yer favor and fortune smile upon ye, brave adventurer! Go forth and claim yer glory, arr!"},
    {"prompt": "How do I learn to code?", "completion": "Start small, ye eager deckhand! Practice every day, debug yer errors like patchin\u2019 a leaky hull, and soon ye\u2019ll command the code seas, arr!"},
    {"prompt": "Describe a sunset.", "completion": "Behold, the sky bleeds gold and crimson as the sun sinks beneath the waves, a sight grand enough to humble any pirate\u2019s heart, arr!"},
    {"prompt": "Give me advice.", "completion": "Trust yer compass, keep yer word, and never abandon yer crew, for loyalty be worth more than all the gold in Davy Jones\u2019 locker, matey!"},
]
print(f'{len(EXAMPLES)} training examples')

18 training examples


## 3. Load GPT-2 and attach LoRA adapters

The base weights are frozen; LoRA adds small trainable matrices to the attention
projection (`c_attn`). Note how few parameters are actually trained.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = 'gpt2'  # GPT-2 small (124M)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['c_attn'], bias='none',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

## 4. Build the dataset

Each example becomes `prompt + completion`. The prompt tokens are masked with
`-100` so the loss is computed on the response only \u2014 teaching the model to
*follow* the instruction format.

In [ ]:
from datasets import Dataset

INSTRUCT_TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n'
MAX_LENGTH = 128

def encode(example):
    prompt_text = INSTRUCT_TEMPLATE.format(instruction=example['prompt'])
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    completion_ids = tokenizer(example['completion'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']

    # Preserve the completion tokens; truncate the prompt from the left if
    # prompt + completion overflows. Avoids all-(-100) rows (no gradient / nan).
    completion_ids = completion_ids[:MAX_LENGTH]
    max_prompt_len = MAX_LENGTH - len(completion_ids)
    if max_prompt_len <= 0:
        prompt_ids = []
    elif len(prompt_ids) > max_prompt_len:
        prompt_ids = prompt_ids[-max_prompt_len:]

    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids

    pad_len = MAX_LENGTH - len(input_ids)
    attention_mask = [1] * len(input_ids) + [0] * pad_len
    input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
    labels = labels + [-100] * pad_len
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

dataset = Dataset.from_list([encode(e) for e in EXAMPLES])
print('Tokenized examples:', len(dataset))

## 5. Train

A handful of epochs on this tiny dataset is enough to see the effect. On a T4 GPU
this is well under a minute.

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

training_args = TrainingArguments(
    output_dir='./gpt2-lora-adapter',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=default_data_collator,
)
trainer.train()
model.save_pretrained('./gpt2-lora-adapter')
print('Adapter saved to ./gpt2-lora-adapter')

## 6. Generate with the fine-tuned model

In [ ]:
def generate(prompt, use_adapter=True, max_new_tokens=80):
    text = INSTRUCT_TEMPLATE.format(instruction=prompt)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    ctx = model.disable_adapter() if not use_adapter else None
    def _gen():
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.8, top_p=0.95,
                                 pad_token_id=tokenizer.pad_token_id)
        return tokenizer.decode(out[0], skip_special_tokens=True)
    if ctx is None:
        return _gen()
    with ctx:
        return _gen()

prompt = 'Tell me about the ocean.'
print('--- FINE-TUNED ---')
print(generate(prompt, use_adapter=True))

## 7. Compare against the untuned base model

`disable_adapter()` temporarily switches off the LoRA layers, giving the original
GPT-2 behavior \u2014 the contrast is the fine-tuning effect.

In [ ]:
print('--- BASE GPT-2 (no adapter) ---')
print(generate(prompt, use_adapter=False))

print()
print('--- FINE-TUNED ---')
print(generate(prompt, use_adapter=True))

## Next steps

- **Your own data:** replace `EXAMPLES` in cell 2 with your prompt/completion pairs.
- **Bigger model:** set `MODEL_NAME = 'gpt2-medium'` (355M) for better quality.
- **Save your adapter:** download the `./gpt2-lora-adapter` folder (a few MB) and
  load it later with `PeftModel.from_pretrained(base_model, adapter_dir)`.
- GPT-2 small is tiny by modern standards \u2014 great for learning the mechanics and
  picking up a style, but it won't reason like a current model.